# 11 — Custom Gymnasium Environments

Building a custom environment is the step that translates a real-world problem into an RL-solvable form. Getting the Gymnasium interface right enables you to use any standard RL library (Stable Baselines3, RLlib, CleanRL) without modification.

In [ ]:
# Custom Gymnasium environment: inventory management
# State: [current_stock, demand_t-1, demand_t-2] (normalised)
# Action: order quantity in [0, max_order]
# Reward: sales revenue - holding cost - stockout penalty

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim

class InventoryEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, max_stock=100, max_order=50, max_demand=30,
                 holding_cost=0.5, stockout_cost=5.0, price=10.0, episode_length=100):
        super().__init__()
        self.max_stock = max_stock; self.max_order = max_order; self.max_demand = max_demand
        self.holding_cost = holding_cost; self.stockout_cost = stockout_cost
        self.price = price; self.episode_length = episode_length

        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(3,), dtype=np.float32)
        self.action_space = spaces.Discrete(max_order + 1)
        self.reset()

    def _obs(self):
        return np.array([self.stock/self.max_stock,
                         self.demand_history[-1]/self.max_demand,
                         self.demand_history[-2]/self.max_demand], dtype=np.float32)

    def _demand(self):
        base = 15
        seasonal = 8 * np.sin(2 * np.pi * self.t / self.episode_length)
        noise = self.np_random.integers(-5, 5)
        return int(np.clip(base + seasonal + noise, 0, self.max_demand))

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.stock = self.max_stock // 2
        self.demand_history = [self.max_demand // 2, self.max_demand // 2]
        self.t = 0
        return self._obs(), {}

    def step(self, action):
        demand = self._demand()
        self.demand_history.append(demand)
        self.stock = min(self.stock + int(action), self.max_stock)
        units_sold = min(self.stock, demand)
        stockout = max(0, demand - self.stock)
        self.stock = max(0, self.stock - demand)
        reward = (units_sold * self.price - self.stock * self.holding_cost - stockout * self.stockout_cost) / 100.0
        self.t += 1
        return self._obs(), reward, self.t >= self.episode_length, False, {"stock": self.stock, "demand": demand}

    def render(self): pass

# ── Validate ──────────────────────────────────────────────────────────────
env = InventoryEnv()
obs, _ = env.reset(seed=42)
print(f"Obs shape: {obs.shape} | dtype: {obs.dtype}")
assert env.observation_space.contains(obs), "Reset obs not in obs space!"

for _ in range(500):
    action = env.action_space.sample()
    obs, r, term, trunc, info = env.step(action)
    assert env.observation_space.contains(obs), f"Step obs out of space: {obs}"
    if term or trunc: obs, _ = env.reset()
print("All step observations within observation_space")

# ── Random baseline ────────────────────────────────────────────────────────
rewards = []
for _ in range(10):
    env2 = InventoryEnv(); env2.reset(); ep_r = 0
    for _ in range(100):
        _, r, term, _, _ = env2.step(env2.action_space.sample())
        ep_r += r
        if term: break
    rewards.append(ep_r)
print(f"Random policy return: mean={np.mean(rewards):.2f}, std={np.std(rewards):.2f}")

# ── Train with REINFORCE ────────────────────────────────────────────────────
class InventoryPolicy(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(3,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh())
        self.actor = nn.Linear(64, 51)
        self.critic = nn.Linear(64, 1)
    def forward(self, x):
        f = self.shared(x)
        return self.actor(f), self.critic(f).squeeze(-1)

net = InventoryPolicy()
optimizer = optim.Adam(net.parameters(), lr=3e-4)
episode_returns = []

for ep in range(500):
    env3 = InventoryEnv(); obs, _ = env3.reset(seed=ep)
    ep_return = 0; transitions = []
    for _ in range(100):
        obs_t = torch.FloatTensor(obs)
        logits, value = net(obs_t)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample(); lp = dist.log_prob(action)
        next_obs, r, term, _, _ = env3.step(action.item())
        transitions.append((obs_t, action, lp, r, value))
        obs = next_obs; ep_return += r
        if term: break
    episode_returns.append(ep_return)
    returns = []; G = 0
    for _,_,_,r,_ in reversed(transitions):
        G = r + 0.99*G; returns.insert(0, G)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    loss = sum(-(lp*(G-v.detach())) + 0.5*(v-G)**2 for (_,_,lp,_,v),G in zip(transitions,returns))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if (ep+1) % 100 == 0:
        print(f"  Ep {ep+1:4d} | Avg return (last 50): {np.mean(episode_returns[-50:]):.2f}")

plt.figure(figsize=(9,4))
w = 30
plt.plot(episode_returns, alpha=0.3, color='steelblue')
plt.plot(np.convolve(episode_returns, np.ones(w)/w, 'valid'), color='steelblue', label=f'{w}-ep avg')
plt.xlabel('Episode'); plt.ylabel('Episode Return')
plt.title('Custom Env: Inventory Management -- REINFORCE', fontweight='bold')
plt.legend(); plt.tight_layout()
plt.savefig('/tmp/inventory_training.png', dpi=100, bbox_inches='tight')
plt.show()


## Env design checklist

- [ ] `observation_space` and `action_space` are correctly typed and bounded
- [ ] `reset(seed=)` is deterministic given the seed
- [ ] `step()` always returns obs within `observation_space`
- [ ] Reward is appropriately scaled (usually [-1, 1] or [-10, 10])
- [ ] Episode terminates within reasonable steps
- [ ] Run `gymcheck` (NB 10) before training